<a href="https://colab.research.google.com/github/claramanolache/DeepLearning/blob/main/CNN_mod2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install opendatasets
!pip install legacy-cgi

#Imports

In [3]:
import numpy as np

from sklearn import datasets
from sklearn import model_selection
from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

import torch
import torch.nn as nn
from torch.utils import data

import opendatasets as od
import pandas as pds

#Importing Dataset

In [46]:
od.download(
    "https://www.kaggle.com/datasets/ikjotsingh221/animal-dataset/data",force=True)

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: claramanolache
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/ikjotsingh221/animal-dataset


100%|██████████| 62.3M/62.3M [00:00<00:00, 131MB/s]


Checking to see what the dataset was downloaded as

In [47]:
import os

folder = "animal-dataset"

print(os.listdir(folder))


['dataset']


In [48]:
import os

train_dir = 'animal-dataset/dataset/Training'
test_dir = 'animal-dataset/dataset/Testing'

print(os.listdir(train_dir))
print(os.listdir(test_dir))

['crows', 'bears', 'rats', 'elephants']
['crows', 'bears', 'rats', 'elephants']


After I get labels from the images, I want to put all the images into one folder so you AI be able to see from file path where it is

In [49]:
from pathlib import Path
subfolder_path = Path("animal-dataset/dataset") / "all_labels"
subfolder_path.mkdir(parents=True, exist_ok=True)
print(f"Subfolder created at: {subfolder_path}")

Subfolder created at: animal-dataset/dataset/all_labels


get labels from root directory and rename. I also want to combine test and train to randomize the division in later step (can't trust how creator seperated)

In [57]:
import uuid
from pathlib import Path
import shutil
import pandas as pd

# target subdirectory for moved files
subfolder_path = Path("animal-dataset/dataset/all_labels")
subfolder_path.mkdir(parents=True, exist_ok=True)

# initialize
df = pd.DataFrame(columns=['file_path', 'label'])

def mover_renamer(directory):
    global df
    new_rows = []
    dir_path = Path(directory)

    label_name = dir_path.name

    for file in dir_path.iterdir():
        if file.is_file():
            extension = file.suffix
            new_filename = f"{uuid.uuid4().hex}{extension}"
            new_path = subfolder_path / new_filename

            shutil.move(str(file), str(new_path))

            new_rows.append({
                'file_path': str(new_path),
                'label': label_name
            })

    if new_rows:
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

# testing with crows:
mover_renamer("animal-dataset/dataset/Training/crows")
display(df.head())


,file_path,label
0,animal-dataset/dataset/all_labels/01fa24632f7f...,crows
1,animal-dataset/dataset/all_labels/48c8f2497937...,crows
2,animal-dataset/dataset/all_labels/e0d469cbca5f...,crows
3,animal-dataset/dataset/all_labels/fcabbed6d5c8...,crows
4,animal-dataset/dataset/all_labels/b498f1ee7f58...,crows


In [58]:
df.shape

(231, 2)

now that I know crows works, I am going to do the rest of the files


In [59]:
mover_renamer("animal-dataset/dataset/Training/bears")
mover_renamer("animal-dataset/dataset/Training/rats")
mover_renamer("animal-dataset/dataset/Training/elephants")
mover_renamer("animal-dataset/dataset/Testing/crows")
mover_renamer("animal-dataset/dataset/Testing/bears")
mover_renamer("animal-dataset/dataset/Testing/rats")
mover_renamer("animal-dataset/dataset/Testing/elephants")

In [61]:
# should be 1564 files
df.shape

(1564, 2)

# Building CNN